
# 🎯 Elasticity Model Optimization (GLM)

**Objective:** Optimize the hyperparameters of a Generalized Linear Model (Negative Binomial) to estimate price elasticity.
**Methodology:**
1. **Feature Engineering:** Cardinality reduction, seasonality extraction, and price centering.
2. **Design Matrix Construction:** Using `RFormula` to handle interactions and dummy variables.
3. **Hyperparameter Tuning:** Using **Optuna** to minimize AIC (Akaike Information Criterion) by tuning ElasticNet regularization parameters (`alpha` and `L1_wt`).

**Input:** `finance_db.gold_sales_events_enriched`


## 📚 1. Setup & Libraries

%pip install optuna==4.0.0
dbutils.library.restartPython()

In [ ]:
import mlflow
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.stats import linregress
from statsmodels.stats.outliers_influence import variance_inflation_factor

# PySpark
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import RFormula

# Configuration
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style="whitegrid", palette="pastel")
import warnings
warnings.filterwarnings('ignore')

# MLflow Setup (Optional: Disable autolog for granular control inside Optuna)
mlflow.autolog(disable=True)


## 📥 2. Data Preparation & Dimensionality Reduction

In [ ]:
# Load Data
df_data = spark.table("finance_db.gold_sales_events_enriched")

In [ ]:
def reduce_cardinality_dynamic(df: DataFrame, cat_cols: dict, volume_col: str = 'BILLED_QTY') -> DataFrame:
    """
    Dynamically groups rare categories into 'Others'.
    
    Args:
        df: Input Spark DataFrame.
        cat_cols: Dictionary where Key = Column Name, Value = Top N categories to keep.
        volume_col: Metric used to determine the 'Top N' (e.g., Sales Volume).
        
    Returns:
        DataFrame with new columns suffixed with '_proc'.
    """
    df_proc = df
    
    for col_name, n_top in cat_cols.items():
        # Identify Top N categories by Volume
        top_categories_rows = (
            df.groupBy(col_name)
            .agg(F.sum(volume_col).alias('total_vol'))
            .orderBy(F.desc('total_vol'))
            .limit(n_top)
            .collect())
        
        top_categories = [row[col_name] for row in top_categories_rows]
        
        # Create processed column
        new_col = f"{col_name}_proc"
        df_proc = df_proc.withColumn(
            new_col,
            F.when(F.col(col_name).isin(top_categories), F.col(col_name))
            .otherwise(F.lit('Others'))).withColumn(new_col, F.col(new_col).cast('string'))
        
        print(f"Processed '{col_name}': Kept top {n_top} categories.")

    return df_proc

In [ ]:
# Configuration for Cardinality Reduction
# Mapping: Column -> Number of Top Categories to keep
cardinality_config = {
    'BRAND': 40,
    'SUB_BUSINESS_UNIT': 15,
    'REGION_STATE': 12,
    'SUB_CHANNEL': 5,
    'SALES_TEAM_CHANNEL': 3,
    'month': 12, # Keep all months if possible, or reduce if data is scarce
    'EAN_CODE': 150 # Keep top 150 SKUs explicitly, group tail
}

# Apply Reduction
df_processed = reduce_cardinality_dynamic(df_data, cardinality_config, volume_col='BILLED_QTY')

# Additional boolean flag reduction
df_processed = df_processed.withColumn(
    'BRAND_CATEGORY_reduced',
    F.when(F.col('BRAND_CATEGORY').cast('int') <= 8, F.lit(1)).otherwise(F.lit(0)))

In [ ]:
# Validation: Check reduction results
for col_orig in cardinality_config.keys():
    col_proc = f"{col_orig}_proc"
    n_orig = df_data.select(col_orig).distinct().count()
    n_proc = df_processed.select(col_proc).distinct().count()
    print(f"{col_orig}: {n_orig} -> {col_proc}: {n_proc}")


## 🛠️ 3. Feature Engineering & Design Matrix

We apply the following transformations:
1. **Log-Log transformation:** `ln_price` vs `log(Q)`.
2. **Seasonality:** Sine/Cosine transformation of `week_of_year`.
3. **Centering:** We center `ln_price` to reduce structural multicollinearity when interacting with categorical variables.
4. **RFormula:** We use Spark's `RFormula` to generate the One-Hot Encoded matrix efficiently.

In [ ]:
import math

def feature_engineering(df: DataFrame, volume_col: str = 'BILLED_QTY') -> DataFrame:
    """
    Creates modeling features: log price, seasonality terms, and target Q.
    """
    return df.withColumn('Q', F.col(volume_col)) \
             .withColumn('ln_price', F.log(F.col('NET_PRICE_UNIT'))) \
             .withColumn(
                 'sin_w',
                 F.sin(2 * F.lit(math.pi) * (F.col('week_of_year') - 1) / 52)) \
             .withColumn(
                 'cos_w',
                 F.cos(2 * F.lit(math.pi) * (F.col('week_of_year') - 1) / 52)) \
             .withColumn('week_of_year', F.col('week_of_year').cast('int'))

# Apply Engineering
df_model_prep = feature_engineering(df_processed, volume_col='BILLED_QTY')

In [ ]:
import re
from typing import Tuple, List

def build_centered_design_matrix(
    df: DataFrame,
    categorical_vars: list,
    seasonal_vars: list,
    volume_col: str = "Q",
    price_col: str = "ln_price",
    price_interaction_exclusion: List[str] = []) -> Tuple[str, DataFrame, float, str]:
    """
    Constructs the design matrix with centered price to mitigate multicollinearity
    in interaction terms.
    
    Returns:
        formula_str: The R-style formula used.
        df_feat: Spark DataFrame with 'features' vector.
        price_mean: Mean value used for centering.
        centered_price_col: Name of the centered column.
    """
    # 1. Center the Price Variable
    price_mean = df.select(F.mean(price_col)).first()[0]
    centered_price_col = f"{price_col}_c"
    df = df.withColumn(centered_price_col, F.col(price_col) - F.lit(price_mean))

    # 2. Ensure Categoricals are Strings
    for c in categorical_vars:
        df = df.withColumn(c, F.col(c).cast("string"))

    # 3. Construct Formula Terms
    # Main Effects
    main_effects = [centered_price_col] + seasonal_vars + categorical_vars
    
    # Interactions (Price * Category)
    interactions = [
        f"{centered_price_col}:{v}" for v in categorical_vars 
        if v not in price_interaction_exclusion] + [
        f"{centered_price_col}:{s}" for s in seasonal_vars
        if s not in price_interaction_exclusion]

    # Formula assembly
    all_terms = main_effects + interactions
    formula_str = f"{volume_col} ~ " + " + ".join(all_terms)

    # 4. Generate Features Vector
    rf = RFormula(
        formula=formula_str,
        featuresCol="features",
        labelCol=volume_col)
    
    # Fit and Transform
    model_rf = rf.fit(df)
    df_feat = model_rf.transform(df)

    return formula_str, df_feat, price_mean, centered_price_col

In [ ]:
# Variables Selection
cat_vars_final = ['BRAND_proc', 
    'UF_proc', # State
    'Sub_Canal_proc', 
    'SALES_TEAM_CHANNEL_proc', 
    'CAT_Marcas_reduced',
    'month_proc',
    'SUB_BUSINESS_UNIT_proc']

seasonal_vars_final = ['sin_w', 'cos_w']

# Filter data (Remove zero sales for log models) and Sampling for Optimization Speed
df_sample = df_model_prep.filter(F.col('Q') > 0).orderBy(F.rand()).limit(20000)

# Build Matrix
formula_used, df_feat_spark, price_mean, price_col_name = build_centered_design_matrix(
     df=df_sample,
     categorical_vars=cat_vars_final,
     seasonal_vars=seasonal_vars_final,
     volume_col='Q',
     price_col='ln_price')

# Convert to Pandas for Optuna/Statsmodels
df_pandas = df_feat_spark.toPandas()

print(f"--- R-Style Formula ---\n{formula_used}\n")
print(f"Dataset Shape: {df_pandas.shape}")


## 🔍 4. Distribution Analysis (Variance Power)
Checking the relationship between Mean and Variance to confirm the choice of the **Negative Binomial** family (where Variance = Mean + alpha * Mean^p).

In [ ]:
# Binning data to estimate Mean-Variance relationship
q_data = df_pandas[['Q']].copy()

# 1. Create bins based on quantiles
stats_agg = (
    q_data.groupby(pd.qcut(q_data['Q'], q=20, duplicates='drop'))['Q']
    .agg(['mean', 'var'])
    .query('var > 0 and mean > 0'))

# 2. Log-Log Regression
log_mean = np.log(stats_agg['mean'])
log_var = np.log(stats_agg['var'])

slope, intercept, r_value, p_value, std_err = linregress(log_mean, log_var)

print(f"Estimated Variance Power (p): {slope:.4f}")

# Interpretation
if 1 < slope < 2:
    print(" Suggests Tweedie distribution (Compound Poisson-Gamma).")
elif slope >= 2:
    print(" Suggests Gamma or Negative Binomial distribution (Overdispersion).")


## 🤖 5. Hyperparameter Optimization (Optuna)

We optimize the **ElasticNet** parameters for the GLM:
* **alpha:** Penalty weight.
* **L1_wt:** The mixing parameter (0 for Ridge, 1 for Lasso).

In [ ]:
def objective(trial):
    """
    Optuna objective function.
    Trains GLM with suggested params and returns AIC.
    """
    # 1. Suggest Hyperparameters
    alpha = trial.suggest_float("alpha", 1e-5, 1e2, log=True)
    l1_wt = trial.suggest_float("L1_wt", 0.0, 1.0)

    try:
        # 2. Define Model (Negative Binomial with Log Link)
        glm_nb = smf.glm(
            formula=formula_used,
            data=df_pandas,
            family=sm.families.NegativeBinomial(link=sm.families.links.log()))

        # 3. Fit Regularized (ElasticNet)
        results_reg = glm_nb.fit_regularized(
            method='elastic_net',
            alpha=alpha,
            L1_wt=l1_wt,
            refit=False # We refit manually below)

        # 4. Refit unregularized using selected features/params to get clean AIC
        # (Statsmodels fit_regularized doesn't calculate AIC directly)
        results = glm_nb.fit(start_params=results_reg.params)
        
        # Metric to Minimize
        aic = results.aic
        
        # Check VIF for multicollinearity (penalty)
        # X_exog = results.model.exog
        # max_vif = np.max([variance_inflation_factor(X_exog, i) for i in range(1, X_exog.shape[1])])
        
        return aic

    except Exception as e:
        # Pruning failed trials
        return float('inf')

# Run Optimization
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50) # Set n_trials higher for production


### 🏆 Optimization Results

In [ ]:
print("--- Optimization Completed ---")
print(f"Best AIC: {study.best_value:,.2f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value:.6f}")


## 📊 6. Visualization of Hyperparameter Search

In [ ]:
# 1. Optimization History (AIC Improvement over trials)
fig1 = optuna.visualization.plot_optimization_history(study)
fig1.update_layout(title="AIC Minimization History")
fig1.show()

# 2. Parameter Importance (What matters most? Alpha or L1_wt?)
fig2 = optuna.visualization.plot_param_importances(study)
fig2.update_layout(title="Hyperparameter Importance")
fig2.show()

# 3. Contour Plot (Interaction between Alpha and L1_wt)
try:
    fig3 = optuna.visualization.plot_contour(study, params=['alpha', 'L1_wt'])
    fig3.update_layout(title="Contour: AIC vs Hyperparameters")
    fig3.show()
except:
    print("Insufficient data for contour plot.")